# Multi-Branch Keras — Modality-Specific Sequence Classifier

This notebook uses a multi-branch deep learning architecture where each sensor modality (Acceleration, Rotation, TOF, Thermopile, STFT, CWT) has its own dedicated branch. 

It follows the exact style of the Random Forest notebook:
- configuration
- data loading
- split
- estimator (Keras Multi-Branch)
- parameter search (grid or Bayesian)
- holdout evaluation
- save results

In [ ]:
import os
import sys
import warnings
import json
import importlib
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import tensorflow as tf

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Local path routing
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# Kaggle path routing
# Kaggle path routing
if os.path.exists('/kaggle/input'):
    for dataset_slug in os.listdir('/kaggle/input'):
        dataset_path = os.path.join('/kaggle/input', dataset_slug)
        if os.path.isdir(dataset_path):
            # Add the root of the dataset
            sys.path.insert(0, dataset_path)
            # If your files are inside a 'src' folder in the dataset, also add that
            src_path_kaggle = os.path.join(dataset_path, 'src')
            if os.path.exists(src_path_kaggle):
                sys.path.insert(0, src_path_kaggle)

# Prefer the exact local source snapshot when this notebook is packaged.
sys.path.insert(0, src_path)

from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline

try:
    import skopt
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

import base_utils as data_utils
import multibranch_utils as multibranch_keras
importlib.reload(multibranch_keras)
BayesSearchCV = multibranch_keras.HierarchicalBayesSearchCV if SKOPT_AVAILABLE else None
from base_utils import (
    SequenceExtractor,
    SensorAugmentor,
    evaluate_holdout,
    prepare_bayesian_space
)
print('Imports loaded successfully')

In [2]:
experiment_name = 'multibranch_attention'

# Supported targets: 'bfrb', 'gesture_action', 'orientation', 'gesture'
TARGET_COL = 'gesture_action'

# Use sample.csv for a quick smoke test; set False for full train.csv
use_sample_data = False
sample_file = 'sample.csv'

search_mode = 'bayesian'  # 'grid' or 'bayesian'
random_state = 42
n_splits = 1              # 1 -> GroupShuffleSplit; >=2 -> GroupKFold
cv_test_size = 0.4
train_size = 0.6
n_iter = 15
verbose = 4
error_score = 'raise'
n_jobs = 1                 # TensorFlow models should not be fitted in parallel

# Leave empty to keep all orientations, especially for orientation prediction.
orientation_filter_list = ['Lie on Back']
problematic_sequence_bool = True

target_only_bool = True

epochs = 5
batch_size = 64

results_dir = Path(workspace_root) / f'results_{experiment_name}_{TARGET_COL}_{search_mode}'
results_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

search_mode = str(search_mode).lower()
if search_mode in ('bayes', 'bayesian'):
    search_mode = 'bayesian'
elif search_mode != 'grid':
    raise ValueError("search_mode must be 'grid' or 'bayesian'")

if n_splits <= 1:
    cv_object = GroupShuffleSplit(
        n_splits=1,
        test_size=cv_test_size,
        random_state=random_state,
    )
else:
    cv_object = GroupKFold(n_splits=n_splits)

In [ ]:
# Load and prepare training data without retaining a second sensor frame.
data_root = data_utils.find_data_root()
sample_path = data_root / sample_file
data_path = sample_path if use_sample_data and sample_path.exists() else data_root / 'train.csv'
sensor_columns = [column for column in pd.read_csv(data_path, nrows=0).columns
                  if column.startswith(('acc_', 'rot_', 'tof_', 'thm_'))]
# Store sensor inputs as float32; signal utilities retain their float64 calculations.
train_df = pd.read_csv(data_path, dtype={column: np.float32 for column in sensor_columns})
print(f'Using {data_path.name}: {train_df["sequence_id"].nunique()} sequences')

train_df['gesture'] = train_df['gesture'].fillna('non_bfrb').astype(str)
train_df['orientation'] = train_df['orientation'].fillna('Unknown').astype(str)
train_df['is_target'] = train_df['sequence_type'].eq('Target').astype(np.uint8)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'].eq(1), 'non_bfrb')
# Parse each distinct gesture once, rather than once for every sensor timestep.
gesture_parts = {gesture: gesture.split(' - ') for gesture in train_df['gesture'].unique()}
train_df['gesture_position'] = train_df['gesture'].map({gesture: parts[0] for gesture, parts in gesture_parts.items()})
train_df['gesture_action'] = train_df['gesture'].map({gesture: parts[-1] for gesture, parts in gesture_parts.items()})

if TARGET_COL not in {'bfrb', 'gesture_action', 'orientation', 'gesture', 'gesture_position'}:
    raise ValueError('Unsupported TARGET_COL.')

# Apply the configured filters in one row selection.
keep_rows = pd.Series(True, index=train_df.index)
if orientation_filter_list:
    keep_rows &= train_df['orientation'].isin(orientation_filter_list)
if target_only_bool:
    keep_rows &= train_df['is_target'].eq(1)
if not keep_rows.all():
    train_df = train_df.loc[keep_rows].copy()
del keep_rows, gesture_parts
train_df[TARGET_COL] = train_df[TARGET_COL].fillna('non_bfrb').astype(str)
print(f'Target: {TARGET_COL}; sequences after filters: {train_df["sequence_id"].nunique()}')


In [ ]:
try:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df,
        train_pct=train_size,
        test_pct=min(0.2, 1 - train_size),
        random_state=random_state,
    )
except Exception as split_error:
    print(f'Balanced split unavailable ({split_error}); using grouped random split.')
    sequence_df = (
        train_df[['sequence_id', 'is_target', TARGET_COL]]
        .drop_duplicates('sequence_id')
        .sort_values('sequence_id')
    )
    splitter = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(sequence_df, groups=sequence_df['sequence_id']))
    train_ids = sequence_df.iloc[train_idx]['sequence_id']
    test_ids = sequence_df.iloc[test_idx]['sequence_id']
    train_sample_df = train_df[train_df['sequence_id'].isin(train_ids)].copy()
    hold_out_df = train_df[train_df['sequence_id'].isin(test_ids)].copy()

X_train = train_sample_df
X_test = hold_out_df
# Keep row-aligned labels for sklearn CV, including available auxiliary targets.
# Raw sensor values and row counters are not sequence labels.
label_columns = [column for column in X_train.columns
                 if column not in sensor_columns and column not in {'row_id', 'sequence_counter'}]
y_train = X_train[label_columns].copy()
y_test = X_test[label_columns].copy()
groups = X_train['sequence_id']

print('Train sequences:', X_train['sequence_id'].nunique())
print('Holdout sequences:', X_test['sequence_id'].nunique())
# The model inputs now own the split frames; release the unsplit sensor data.
del train_df, train_sample_df, hold_out_df


In [5]:
extractor = SequenceExtractor(
    output_format='chunks',
    acc_modes='smoothed|velocity|displacement|jerk',
    rotation_modes='quaternion|euler|angular_velocity',
    tof_modes='pooled_stats|sensor_stats',
    thm_modes='centered_diff',
    frame_stats='mean,std,min,max,last,first,rms',
)

clf = multibranch_keras.MultiBranchSequenceClassifier(
    extractor=extractor,
    primary_target=TARGET_COL,
    epochs=epochs,
    batch_size=batch_size,
    random_state=random_state,
)

pipeline = Pipeline([
    ('augmentor', SensorAugmentor(
        sequence_col='sequence_id',
        counter_col='sequence_counter',
        prob=0.0,
        per_aug_prob=0.0,
        seed=random_state,
    )),
    ('estimator', clf),
])

In [6]:
SEARCH_BRANCH_FILTERS = [
    {'acc': 16, 'rotation': 16, 'tof': 16, 'thermo': 16, 'stft': 16, 'cwt': 16},
    {'acc': 32, 'rotation': 32, 'tof': 32, 'thermo': 32, 'stft': 16, 'cwt': 16},
    {'acc': 64, 'rotation': 64, 'tof': 64, 'thermo': 32, 'stft': 32, 'cwt': 32},
    {'acc': 128, 'rotation': 128, 'tof': 128, 'thermo': 32, 'stft': 32, 'cwt': 32},
    {'acc': 256, 'rotation': 256, 'tof': 256, 'thermo': 32, 'stft': 32, 'cwt': 32},
]

SEARCH_BRANCH_DEPTHS = [
    {'acc': 2, 'rotation': 2, 'tof': 2, 'thermo': 1, 'stft': 1, 'cwt': 1},
    {'acc': 3, 'rotation': 3, 'tof': 2, 'thermo': 1, 'stft': 1, 'cwt': 1},
    {'acc': 3, 'rotation': 3, 'tof': 3, 'thermo': 2, 'stft': 2, 'cwt': 2},
]

# Explicit per-layer schedules. This example gives acc a 32-64-64 stack
# while keeping thermo as a single 32-filter layer.
SEARCH_CUSTOM_FILTERS = [
    {
        'acc': [32, 64, 64],
        'rotation': [32, 64, 64],
        'tof': [32, 64],
        'thermo': [32],
        'stft': [32],
        'cwt': [32],
    },
]

SEARCH_CONSTANT_FILTERS = [
    {'acc': 64, 'rotation': 64, 'tof': 64, 'thermo': 32, 'stft': 32, 'cwt': 32},
    {'acc': 128, 'rotation': 128, 'tof': 128, 'thermo': 32, 'stft': 32, 'cwt': 32},
]

# Select one compatible filter representation for this search.
branch_filter_mode = 'custom'  # 'custom', 'double', or 'constant'
if branch_filter_mode == 'custom':
    branch_filter_candidates = SEARCH_CUSTOM_FILTERS
    branch_depth_candidates = [{
        'acc': 3, 'rotation': 3, 'tof': 2,
        'thermo': 1, 'stft': 1, 'cwt': 1,
    }]
else:
    if branch_filter_mode == 'double':
        branch_filter_candidates = SEARCH_BRANCH_FILTERS
    elif branch_filter_mode == 'constant':
        branch_filter_candidates = SEARCH_CONSTANT_FILTERS
    else:
        raise ValueError("branch_filter_mode must be 'custom', 'double', or 'constant'.")
    branch_depth_candidates = SEARCH_BRANCH_DEPTHS

# Example stacking patterns for STFT/CWT.
# If you set stft_configs / cwt_configs to a list of dicts, the scalar params below
# are effectively the defaults for the default extractor and are therefore redundant
# with the per-config values you pass in the stacked list.
STFT_CONFIGS = [
    None,
    [{'name': 'stft_default', 'nperseg': 64, 'noverlap': 32}],
    [
        {'name': 'stft_fast', 'nperseg': 32, 'noverlap': 16},
        {'name': 'stft_mid', 'nperseg': 64, 'noverlap': 32},
        {'name': 'stft_slow', 'nperseg': 128, 'noverlap': 64},
    ],
]

CWT_CONFIGS = [
    None,
    [{'name': 'cwt_morl', 'wavelet': 'morl', 'max_scale': 64, 'n_scales': 32}],
    [
        {'name': 'cwt_morl', 'wavelet': 'morl', 'max_scale': 64, 'n_scales': 32},
        {'name': 'cwt_mexh', 'wavelet': 'mexh', 'max_scale': 64, 'n_scales': 32},
        {'name': 'cwt_gaus2', 'wavelet': 'gaus2', 'max_scale': 64, 'n_scales': 32},
    ],
]

GRID_PARAM_SPACE = {
    'augmentor__prob': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__per_aug_prob': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__sensor_drop_prob': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__noise_std': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__jitter_sigma': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__scaling_sigma': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__channel_drop_prob': [0.0, 0.1, 0.3, 0.5, 0.6],
    'augmentor__time_shift_frac': [0.0, 0.1, 0.2],
    'augmentor__temporal_num_masks': [0, 1, 2],
    'augmentor__temporal_mask_frac': [0.0, 0.1, 0.2],
    'estimator__extractor__acc_modes': [None, 'smoothed|velocity|displacement|jerk'],
    'estimator__extractor__rotation_modes': [None, 'quaternion|euler|angular_velocity'],
    'estimator__extractor__tof_modes': [None, 'pooled_stats|sensor_stats'],
    'estimator__extractor__thm_modes': [None, 'centered_diff'],
    'estimator__extractor__frame_stats': [None, 'mean,std,min,max,last,first,rms'],
    'estimator__extractor__motion_filter_mode': [None],
    'estimator__extractor__use_dead_reckoning': [False],
    'estimator__extractor__dead_reckoning_detrend': [False],
    'estimator__extractor__kalman_process_noise': [1e-3],
    'estimator__extractor__kalman_measurement_noise': [1e-1],
    'estimator__extractor__window_size': [5, 10],
    'estimator__extractor__smooth_alpha': [None],
    'estimator__extractor__clip_value': [None],
    'estimator__extractor__interp_mode': ['linear'],
    'estimator__extractor__output_format': ['chunks'],
    'estimator__extractor__padding_value': [0.0],
    'estimator__extractor__maxlen': [160],
    'estimator__extractor__chunk_window_size': [100],
    'estimator__extractor__chunk_stride': [50],
    'estimator__extractor__add_global_context': [True],
    'estimator__extractor__resample_modalities': [False, True],
    'estimator__extractor__compute_dt': [False, True],
    'estimator__extractor__imu_native_sampling_rate': [100],
    'estimator__extractor__rot_native_sampling_rate': [100],
    'estimator__extractor__tof_native_sampling_rate': [20],
    'estimator__extractor__thm_native_sampling_rate': [20],
    'estimator__extractor__imu_target_sampling_rate': [100],
    'estimator__extractor__rot_target_sampling_rate': [100],
    'estimator__extractor__tof_target_sampling_rate': [20],
    'estimator__extractor__thm_target_sampling_rate': [20],
    'estimator__extractor__stft_configs': STFT_CONFIGS,
    'estimator__extractor__cwt_configs': CWT_CONFIGS,
    'estimator__extractor__stft_use_log_scale': [True],
    'estimator__extractor__cwt_use_log_scale': [True],
    'estimator__extractor__filter_problematic_sequences': [problematic_sequence_bool],
    'estimator__extractor__ideal_skew_threshold': [1.0, 1.1, 1.2, 1.3, 1.4, 1.5],
    'estimator__extractor__problematic_features_threshold': [4, 5, 6, 7],
    'estimator__branch_filters': branch_filter_candidates,
    'estimator__branch_num_conv_layers': branch_depth_candidates,
    'estimator__branch_filter_mode': [branch_filter_mode],
    'estimator__conv_dropout': [0.0, 0.1, 0.2, 0.5],
    'estimator__static_dropout': [0.0, 0.1, 0.2, 0.5],
    'estimator__dense_units': [32, 64, 128],
    'estimator__dropout': [0.2, 0.4, 0.5],
    'estimator__validation_split': [0.15],
    'estimator__early_stopping_patience': [3, 10],
    'estimator__learning_rate': [1e-3, 1e-4],
    'estimator__epochs': [50, 100],
    'estimator__batch_size': [32, 64, 128],
}

if SKOPT_AVAILABLE:
    BAYESIAN_PARAM_SPACE = multibranch_keras.prepare_multibranch_bayesian_space(
        GRID_PARAM_SPACE
    )
else:
    BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE

for space_name, candidate_space in (
    ('grid', GRID_PARAM_SPACE),
    ('bayesian', BAYESIAN_PARAM_SPACE),
):
    unknown_params = set(candidate_space) - set(pipeline.get_params(deep=True))
    if unknown_params:
        raise ValueError(
            f'Unknown {space_name} search parameters: {sorted(unknown_params)}'
        )

param_space = BAYESIAN_PARAM_SPACE if search_mode == 'bayesian' else GRID_PARAM_SPACE
print(f'Validated {len(GRID_PARAM_SPACE)} Grid and Bayesian search parameters.')

Validated 59 Grid and Bayesian search parameters.


In [ ]:
if search_mode == 'bayesian':
    if not SKOPT_AVAILABLE:
        raise ImportError(
            "Bayesian search requires scikit-optimize. "
            "Install with: pip install scikit-optimize"
        )
    search = BayesSearchCV(
        estimator=pipeline,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=None,  # Pipeline delegates to sequence-level estimator.score().
        cv=cv_object,
        n_jobs=n_jobs,
        pre_dispatch='n_jobs',
        random_state=random_state,
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )
else:
    search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_space,
        scoring=None,
        cv=cv_object,
        n_jobs=n_jobs,
        pre_dispatch='n_jobs',
        verbose=verbose,
        return_train_score=True,
        error_score=error_score,
    )

search.fit(X_train, y_train, groups=groups)

print('Best CV score:', search.best_score_)
print('Best params:', search.best_params_)

In [8]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

eval_results = evaluate_holdout(
    y_test,
    y_pred,
    target_col=TARGET_COL,
    verbose=True,
)

print('Holdout competition score / macro F1:', eval_results['competition_score'])


FINAL EVALUATION
Target: gesture_action
Macro F1: 0.4486

----------------------------------------
Classification Report
----------------------------------------
               precision    recall  f1-score   support

   pinch skin       0.55      0.60      0.57       127
    pull hair       0.71      0.24      0.36       199
pull hairline       0.25      0.86      0.39        58
      scratch       0.53      0.43      0.48       125

     accuracy                           0.45       509
    macro avg       0.51      0.53      0.45       509
 weighted avg       0.57      0.45      0.44       509

Holdout competition score / macro F1: 0.4485867393903156


In [9]:
save_dir = results_dir / timestamp
save_dir.mkdir(parents=True, exist_ok=True)
best_estimator = best_model.named_steps['estimator']
best_estimator.model_.save(save_dir / 'best_model.h5')

pd.DataFrame(search.cv_results_).to_csv(
    save_dir / f'{experiment_name}_cv.csv',
    index=False,
)
eval_results['results_df'].to_csv(
    save_dir / f'{experiment_name}_holdout.csv',
    index=False,
)
pd.DataFrame([{
    'target': TARGET_COL,
    'best_score': search.best_score_,
    'best_params': json.dumps(search.best_params_, default=str),
    'holdout_score': eval_results['competition_score'],
}]).to_csv(
    save_dir / f'{experiment_name}_best.csv',
    index=False,
)

with open(save_dir / 'results.json', 'w') as results_file:
    json.dump({
        'target': TARGET_COL,
        'best_score': float(search.best_score_),
        'best_params': search.best_params_,
        'holdout_score': float(eval_results['competition_score']),
        'random_state': random_state,
        'train_sequences': int(X_train['sequence_id'].nunique()),
        'holdout_sequences': int(X_test['sequence_id'].nunique()),
    }, results_file, indent=2, default=str)

print(f'Results saved to {save_dir}')

Results saved to /kaggle/working/results_multibranch_attention_gesture_action_bayesian/20260928_052350
